In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 4.7 MB/s eta 0:00:00


In [2]:
import cv2
import csv
import os
import glob
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pycocotools import mask as mask_utils
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [3]:
class Config_YOLO():
    TEST_IMAGES_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"
    MODEL_PATH = "/kaggle/input/datasets/hdjojo/solar-filament-seg-models/magfilo_260825_yolov8l_seg_imgsz2048_nc1.pt"
    IMG_SIZE = 2048
    CONF = 0.3
    IOU = 0.0
    MAX_DET = 100
    
CONFIG_YOLO = Config_YOLO()
model_yolo = YOLO(CONFIG_YOLO.MODEL_PATH)

In [4]:
def mask_to_rle(binary_mask):
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle["counts"]

def remove_instance_overlaps(masks, scores):
    
    sorted_indices = np.argsort(scores)[::-1]
    sorted_masks = masks[sorted_indices]
    
    accumulated_mask = np.zeros(masks.shape[1:], dtype=np.uint8)
    non_overlapping_masks = []
    
    for mask in sorted_masks:
        clean_mask = mask & ~accumulated_mask
        non_overlapping_masks.append(clean_mask)
        accumulated_mask |= mask
        
    return np.array(non_overlapping_masks)

In [5]:
submission_rows = []
for img_path in tqdm(sorted(glob.glob(f"{CONFIG_YOLO.TEST_IMAGES_DIR}/*.jpeg")), desc="Inference"):
    file_stem = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    result = model_yolo.predict(
        img_path, imgsz=CONFIG_YOLO.IMG_SIZE, conf=CONFIG_YOLO.CONF, iou=CONFIG_YOLO.IOU, max_det=CONFIG_YOLO.MAX_DET, verbose=False)[0]
    
    if result.masks is None:
        #empty_mask = np.zeros((2048, 2048), dtype=np.uint8)
        #empty_rle_str = mask_to_rle(empty_mask)
        #submission_rows.append({
        #    "filament_id": f"{file_stem}_1",
        #    "segmentation_rle": empty_rle_str
        #})
        continue

    result_masks_data = result.masks.data.cpu().numpy()
    scores = result.boxes.cls.cpu().numpy()
    new_masks_data = remove_instance_overlaps(result_masks_data, scores)
    
    for i, m in enumerate(new_masks_data):
        m_res = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        if m_res.sum() < 5: continue
        submission_rows.append({
            "filament_id": f"{file_stem}_{i+1}",
            "segmentation_rle": mask_to_rle(m_res)
        })

Inference: 100%|██████████| 180/180 [02:30<00:00,  1.19it/s]


In [6]:
sub_path = "/kaggle/working/submission.csv"
with open(sub_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filament_id", "segmentation_rle"])
    writer.writeheader()
    writer.writerows(submission_rows)
print(f"Submission saved to {sub_path} ({len(submission_rows)} instances)")

Submission saved to /kaggle/working/submission.csv (1342 instances)
